── Data source ───────────────────────────────────────────────────────────────
MODE = 'local'   : use small test CSVs already in swapped_data/ (10 sims)
                   good for layout / style iteration; saves figure as *_mockup.pdf
MODE = 'cluster' : use pre-averaged CSVs computed on the cluster and rsync'd here
                   set CLUSTER_TIER to match which run you ported over

To pull cluster CSVs (run from repo root):
  rsync -av sherlock:$SCRATCH/misperception/swapped_data_cluster_full/ \
      06_swap_simulations/swapped_data_cluster_full/
──────────────────────────────────────────────────────────────────────────────

In [ ]:
MODE         = 'cluster'   # 'local' | 'cluster'
CLUSTER_TIER = 'full'    # 'quick' | 'medium' | 'full' — ignored when MODE == 'local'

if MODE == 'local':
    DATA_DIR    = 'swapped_data'
    CSV_PATTERN = 'swap_sim_hss_{hss}_hoo_{hoo}.csv'
    fig_suffix  = 'mockup'
else:
    DATA_DIR    = f'swapped_data_cluster_{CLUSTER_TIER}'
    CSV_PATTERN = 'swap_sim_hss_{hss}_hoo_{hoo}_m5.csv'
    fig_suffix  = CLUSTER_TIER

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.lines import Line2D
import sys

In [ ]:
from pyprojroot import here

In [ ]:
colors    = ['#19bdff', '#f2d138']  # majority (support), minority (oppose)
greycolor = '#626262'
myblack   = '#222222'

plt.rc('font', **{'family': 'sans-serif', 'sans-serif': ['Arial']})
plt.rcParams['font.family'] = 'Arial'

SMALL_SIZE  = 9
MEDIUM_SIZE = 9
BIGGER_SIZE = 10

plt.rc('font',   size=SMALL_SIZE)
plt.rc('axes',   titlesize=SMALL_SIZE)
plt.rc('axes',   labelsize=MEDIUM_SIZE)
plt.rc('xtick',  labelsize=SMALL_SIZE)
plt.rc('ytick',  labelsize=SMALL_SIZE)
plt.rc('legend', fontsize=SMALL_SIZE)
plt.rc('figure', titlesize=BIGGER_SIZE)

def customize_plot(axis, color='black'):
    axis.spines['bottom'].set_color(color)
    axis.spines['left'].set_color(color)
    axis.tick_params(axis='x', colors=color, labelsize=SMALL_SIZE)
    axis.tick_params(axis='y', colors=color, labelsize=SMALL_SIZE)
    sns.despine(ax=axis)

In [ ]:
CONDITIONS = [
    (0.25, 0.25),
    (0.50, 0.50),
    (0.25, 0.75),
    (0.75, 0.25),
    (0.50, 0.75),
    (0.75, 0.50),
]

# Grid layout: 3 rows × 2 cols
#   Row 0 — symmetric homophily
#   Row 1 — higher homophily among opponents (h_oo > h_ss)
#   Row 2 — higher homophily among supporters (h_ss > h_oo)
LAYOUT = [
    (0.25, 0.25), (0.50, 0.50),
    (0.25, 0.75), (0.50, 0.75),
    (0.75, 0.25), (0.75, 0.50),
]

num_agents       = 1000
majority_percent = (1 - 1/3) * 100   # 66.67
dot_size         = 8

# Fixed empirical beta_overall range (percentage points)
BETA_EMPIRICAL_LO = 23
BETA_EMPIRICAL_HI = 29


In [ ]:
dfs = {}
for h_ss, h_oo in CONDITIONS:
    hss_str  = str(int(h_ss * 100))
    hoo_str  = str(int(h_oo * 100))
    filename = CSV_PATTERN.format(hss=hss_str, hoo=hoo_str)
    csv_path = str(here() / "06_swap_simulations" / DATA_DIR / filename)
    if os.path.exists(csv_path):
        dfs[(h_ss, h_oo)] = pd.read_csv(csv_path)
        print(f'Loaded  {csv_path}')
    else:
        print(f'MISSING {csv_path}')

In [ ]:
def empirical_xspan(x_pct, beta_overall):
    """Return (x_entry, x_exit) where beta_overall is inside [LO, HI], or None."""
    inside = (beta_overall >= BETA_EMPIRICAL_LO) & (beta_overall <= BETA_EMPIRICAL_HI)
    if not inside.any():
        return None
    x_entry = x_pct[np.argmax(inside)]
    x_exit  = x_pct[len(inside) - 1 - np.argmax(inside[::-1])]
    return x_entry, x_exit

In [ ]:
ncols = 2
nrows = 3
fig, axes = plt.subplots(nrows, ncols, figsize=(3.42, 6.0), dpi=300)
axes_flat = axes.flatten()

row_labels = [
    'Symmetric homophily',
    'Higher opponent homophily',
    'Higher supporter homophily',
]

handles = [
    Line2D([0], [0], color='none', marker='s', markersize=7, label='overall',
           markerfacecolor=greycolor, markeredgewidth=0),
    Line2D([0], [0], color='none', marker='s', markersize=7, label='support',
           markerfacecolor=colors[0], markeredgewidth=0),
    Line2D([0], [0], color='none', marker='s', markersize=7, label='oppose',
           markerfacecolor=colors[1], markeredgewidth=0),
]

panel_labels = list('ABCDEF')

# ── Scatter panels ─────────────────────────────────────────────────────────
for idx, key in enumerate(LAYOUT):
    ax  = axes_flat[idx]
    col = idx % ncols

    df    = dfs[key]
    x_pct = df['swap_count'].values * 100 / num_agents

    beta_overall_pct = majority_percent - df['mean_opinion_percent'].values
    beta_maj_pct     = majority_percent - df['mean_majority_opinion_percent'].values
    beta_min_pct     = majority_percent - df['mean_minority_opinion_percent'].values

    beta_overall = beta_overall_pct / 100
    beta_maj     = beta_maj_pct     / 100
    beta_min     = beta_min_pct     / 100

    ax.scatter(x_pct, beta_overall, color=greycolor, s=dot_size, label='overall',     zorder=2)
    ax.scatter(x_pct, beta_maj,     color=colors[0], s=dot_size, label='support', zorder=2)
    ax.scatter(x_pct, beta_min,     color=colors[1], s=dot_size, label='oppose',  zorder=2)

    span = empirical_xspan(x_pct, beta_overall_pct)
    if span is not None:
        ax.axvspan(span[0], span[1], color='#a83232', alpha=0.3, edgecolor=None, zorder=1)

    if key == (0.25, 0.25):
        ax.legend(handles=handles, loc='lower right', frameon=False,
                  fontsize=SMALL_SIZE, labelspacing=0.15,
                  bbox_to_anchor=(1.0, -0.08))

    ax.text(-0.075, 1.02, panel_labels[idx], transform=ax.transAxes,
            fontsize=SMALL_SIZE, fontweight='bold', va='bottom', ha='left')

    h_ss, h_oo = key
    ax.set_xlim(0, 33)
    ax.set_ylim(-0.33, 0.66)
    ax.set_xticks([0, 10, 20, 30])
    ax.set_xlabel('% nodes swapped', size=SMALL_SIZE)
    if col == 0:
        ax.set_ylabel(r'misperception $\beta$', size=SMALL_SIZE)
    if col == 1:
        ax.tick_params(labelleft=False)
    ax.set_title(f'$h_{{ss}}$={h_ss}, $h_{{oo}}$={h_oo}', size=SMALL_SIZE)
    customize_plot(ax)

# ── Layout ────────────────────────────────────────────────────────────────
plt.tight_layout(rect=[0.0, 0.0, 1.0, 1.0])
plt.subplots_adjust(hspace=1.0, wspace=0.1)

# ── Suptitle centered over the axes area ──────────────────────────────────
fig.canvas.draw()
pos0 = axes[0, 0].get_position()
pos1 = axes[0, 1].get_position()
x_mid = (pos0.x0 + pos1.x1) / 2
fig.text(x_mid, 1.05, 'Interaction of asymmetric homophily\nand simulated media bias',
         ha='center', va='bottom', fontsize=SMALL_SIZE, transform=fig.transFigure)

# ── Row labels: centered across both columns in figure coords ─────────────
for row, label in enumerate(row_labels):
    pos0 = axes[row, 0].get_position()
    pos1 = axes[row, 1].get_position()
    x_mid = (pos0.x0 + pos1.x1) / 2
    y_top = pos0.y1 + 0.055
    fig.text(x_mid, y_top, label, ha='center', va='bottom',
             fontsize=SMALL_SIZE, transform=fig.transFigure)

out_path = str(here() / "figures" / f'figure_asymmetric_swaps_{fig_suffix}.pdf')
plt.savefig(out_path, format='pdf', bbox_inches='tight')
plt.show()
print(f'Saved {out_path}')

In [ ]:
from matplotlib.patches import Polygon as MplPolygon

fig_diag, ax_diag = plt.subplots(1, 1, figsize=(2.5, 2.5), dpi=300)

upper = MplPolygon([(0,0),(0,1),(1,1)], closed=True,
                   facecolor=colors[1], alpha=0.12, edgecolor='none', zorder=0)
lower = MplPolygon([(0,0),(1,0),(1,1)], closed=True,
                   facecolor=colors[0], alpha=0.12, edgecolor='none', zorder=0)
ax_diag.add_patch(upper)
ax_diag.add_patch(lower)

# Include (0.75, 0.75) even though it is not in the scatter figure
DIAGRAM_CONDITIONS = CONDITIONS + [(0.75, 0.75)]
dfs_diag = dict(dfs)
for h_ss, h_oo in DIAGRAM_CONDITIONS:
    if (h_ss, h_oo) not in dfs_diag:
        hss_str  = str(int(h_ss * 100))
        hoo_str  = str(int(h_oo * 100))
        filename = CSV_PATTERN.format(hss=hss_str, hoo=hoo_str)
        csv_path = str(here() / "06_swap_simulations" / DATA_DIR / filename)
        if os.path.exists(csv_path):
            dfs_diag[(h_ss, h_oo)] = pd.read_csv(csv_path)

for h_ss, h_oo in DIAGRAM_CONDITIONS:
    if (h_ss, h_oo) not in dfs_diag:
        label, txt_color, fsize = '?', greycolor, 7
    else:
        df    = dfs_diag[(h_ss, h_oo)]
        x_pct = df['swap_count'].values * 100 / num_agents
        beta  = majority_percent - df['mean_opinion_percent'].values
        span  = empirical_xspan(x_pct, beta)
        if span is None:
            label     = '×' if beta.min() > BETA_EMPIRICAL_HI else '—'
            txt_color = myblack
            fsize     = 9
        else:
            label     = f'{span[0]:.0f}%'
            txt_color = myblack
            fsize     = 7
    ax_diag.text(h_ss, h_oo, label,
                 ha='center', va='center',
                 fontsize=fsize, color=txt_color, fontweight='bold', zorder=4)

ax_diag.plot([0, 1], [0, 1], color=greycolor, linestyle='--', linewidth=0.8, zorder=2)
ax_diag.set_xlim(0, 1)
ax_diag.set_ylim(0, 1)
ax_diag.set_xticks([0.25, 0.5, 0.75])
ax_diag.set_yticks([0.25, 0.5, 0.75])
ax_diag.set_xlabel('$h_{ss}$', size=SMALL_SIZE)
ax_diag.set_ylabel('$h_{oo}$', size=SMALL_SIZE)
ax_diag.set_aspect('equal', adjustable='box')
customize_plot(ax_diag)

ax_diag.set_title(
    'Minimal percent of node swaps required\nto reach empirical misperception range',
    size=SMALL_SIZE - 1, pad=6
)

plt.tight_layout()

out_path_diag = str(here() / "figures" / "figure_asymmetric_swaps_diagram.pdf")
fig_diag.savefig(out_path_diag, format='pdf', bbox_inches='tight')
plt.show()
print(f'Saved {out_path_diag}')
